# Offline vs Online Evaluation

Agent quality is measured in **two places**:

| Mode | When | Data | Goal |
|------|------|------|------|
| **Offline** | Pre-deploy (CI, notebooks) | Golden dataset, frozen snapshots | Block regressions |
| **Online** | Production traffic | Live user queries, traces, feedback | Detect drift, catch edge cases |

```
  OFFLINE (CI)                         ONLINE (prod)
  ────────────                         ─────────────
  golden set ──► eval harness          live traces ──► sampled eval
       │                                      │
       ▼                                      ▼
  pass rate gate                       drift alerts + user thumbs
       │                                      │
       └──────────► golden set ◄──────────────┘
                  (feedback loop)
```

Offline eval is **fast and deterministic**. Online eval is **representative of reality**. You need both.

This notebook implements:

1. Offline CI harness (golden set + regression gate)
2. Online monitor (trace sampling, feedback, drift detection)
3. Shadow mode — compare candidate vs production without user impact
4. Feedback loop — promote production failures to golden set

Plain Python, offline by default.

In [ ]:
"""
ShopCo Support — offline vs online evaluation lab.
"""

import json
import os
import random
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from enum import Enum
from typing import Any, Literal

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days. Refunds in 5-7 business days. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing"},
}

print("ShopCo offline/online eval lab ready.")

---

## 1. Why Both Modes Matter

**Offline only** — you ship with confidence on known cases but miss real-world phrasing, data drift, and seasonal spikes.

**Online only** — you react to problems after users suffer; no pre-deploy safety net.

**Together** — offline gates releases; online catches what goldens never anticipated.

---

## 2. Comparison Matrix

In [ ]:
OFFLINE_VS_ONLINE = [
    ("Timing", "Pre-deploy / CI", "Continuous in production"),
    ("Data", "Curated golden set", "Live user traffic"),
    ("Cost", "Low (batch, cached)", "Higher (sampling + judges)"),
    ("Determinism", "High", "Variable (live DB, LLM)"),
    ("Latency budget", "Minutes OK", "Must be async / sampled"),
    ("Best for", "Regressions, safety gates", "Drift, new intents, satisfaction"),
]

print(f"{'Dimension':<16} {'Offline':<22} {'Online'}")
print("-" * 60)
for dim, off, on in OFFLINE_VS_ONLINE:
    print(f"{dim:<16} {off:<22} {on}")

---

## 3. Shared Agent + Trace Model

In [ ]:
@dataclass
class AgentTrace:
    trace_id: str
    user_id: str
    session_id: str
    question: str
    answer: str
    intent: str
    tools_called: list[str]
    citations: list[str]
    latency_ms: float
    timestamp: str
    agent_version: str = "v1.0.0"
    error: str = ""


class ShopCoAgent:
    def __init__(self, version: str = "v1.0.0", *, broken: bool = False) -> None:
        self.version = version
        self.broken = broken  # simulate regression for demo

    def invoke(self, question: str, user_id: str = "user", session_id: str = "sess") -> AgentTrace:
        import time
        t0 = time.perf_counter()
        q = question.lower()
        intent, tools, cites, answer = "general", [], [], ""

        if self.broken and "return" in q:
            answer = "Returns are not available."  # regression: wrong policy
            intent = "policy"
        elif "return" in q or "refund" in q:
            intent, tools, cites = "policy", ["retrieve_policy"], ["pol-returns-01"]
            answer = POLICY_SNIPPETS["returns"]
        elif "order" in q or re.search(r"ORD-[0-9]{4}", question.upper()):
            intent, tools = "order", ["lookup_order"]
            m = re.search(r"ORD-[0-9]{4}", question.upper())
            oid = m.group(0) if m else "ORD-1001"
            answer = f"Order {oid} status: {ORDERS_DB.get(oid, {}).get('status', 'unknown')}."
        elif "shipping" in q:
            intent, tools, cites = "policy", ["retrieve_policy"], ["pol-shipping-02"]
            answer = POLICY_SNIPPETS["shipping"]
        else:
            answer = "How can I help with your ShopCo order or policy?"

        return AgentTrace(
            trace_id=f"tr-{uuid.uuid4().hex[:8]}",
            user_id=user_id, session_id=session_id, question=question,
            answer=answer, intent=intent, tools_called=tools, citations=cites,
            latency_ms=(time.perf_counter() - t0) * 1000,
            timestamp=utc_now().isoformat(), agent_version=self.version,
        )


PROD_AGENT = ShopCoAgent(version="v1.0.0")
print(PROD_AGENT.invoke("Return policy?").answer[:60])

---

## 4. Offline Evaluation — Golden Set + CI Gate

In [ ]:
@dataclass
class GoldenCase:
    id: str
    input: str
    must_contain: list[str] = field(default_factory=list)
    must_cite: list[str] = field(default_factory=list)
    expected_intent: str = ""
    tags: list[str] = field(default_factory=list)


GOLDEN_SET: list[GoldenCase] = [
    GoldenCase("pol-001", "Return window?", must_contain=["30"], must_cite=["pol-returns-01"], expected_intent="policy", tags=["policy"]),
    GoldenCase("ord-001", "ORD-1001 status", must_contain=["shipped"], expected_intent="order", tags=["order"]),
    GoldenCase("ship-001", "Free shipping threshold?", must_contain=["50"], must_cite=["pol-shipping-02"], expected_intent="policy", tags=["policy"]),
]

OFFLINE_BASELINE = {"pass_rate": 1.0, "policy": 1.0}


@dataclass
class OfflineEvalResult:
    case_id: str
    passed: bool
    reasons: list[str]


@dataclass
class OfflineReport:
    run_id: str
    agent_version: str
    pass_rate: float
    results: list[OfflineEvalResult]
    gate_passed: bool


def eval_offline_case(trace: AgentTrace, case: GoldenCase) -> OfflineEvalResult:
    reasons: list[str] = []
    for term in case.must_contain:
        if term.lower() not in trace.answer.lower():
            reasons.append(f"missing '{term}'")
    for cite in case.must_cite:
        if cite not in trace.citations and cite not in trace.answer:
            reasons.append(f"missing citation {cite}")
    if case.expected_intent and trace.intent != case.expected_intent:
        reasons.append(f"intent {trace.intent} != {case.expected_intent}")
    return OfflineEvalResult(case.id, not reasons, reasons)


def run_offline_eval(agent: ShopCoAgent, cases: list[GoldenCase], baseline: dict[str, float]) -> OfflineReport:
    results = []
    for case in cases:
        trace = agent.invoke(case.input)
        results.append(eval_offline_case(trace, case))
    pass_rate = sum(1 for r in results if r.passed) / len(results) if results else 0.0
    gate = pass_rate >= baseline.get("pass_rate", 0.0)
    return OfflineReport(
        run_id=f"offline-{uuid.uuid4().hex[:6]}",
        agent_version=agent.version, pass_rate=pass_rate,
        results=results, gate_passed=gate,
    )


offline_ok = run_offline_eval(PROD_AGENT, GOLDEN_SET, OFFLINE_BASELINE)
print(f"Offline {offline_ok.run_id}: {offline_ok.pass_rate:.0%} pass, gate={'OPEN' if offline_ok.gate_passed else 'BLOCKED'}")

---

## 5. Demo — Offline Gate Blocks Bad Release

In [ ]:
BROKEN_AGENT = ShopCoAgent(version="v1.1.0-broken", broken=True)
offline_bad = run_offline_eval(BROKEN_AGENT, GOLDEN_SET, OFFLINE_BASELINE)

print(f"Broken agent v{BROKEN_AGENT.version}: pass_rate={offline_bad.pass_rate:.0%}")
for r in offline_bad.results:
    if not r.passed:
        print(f"  ✗ {r.case_id}: {r.reasons}")
print(f"Deploy gate: {'ALLOW' if offline_bad.gate_passed else 'BLOCK'}")

---

## 6. Online Evaluation — Production Trace Monitor

In [ ]:
@dataclass
class UserFeedback:
    trace_id: str
    score: float  # 0.0 thumbs down, 1.0 thumbs up
    comment: str = ""


@dataclass
class OnlineEvalResult:
    trace_id: str
    auto_score: float
    feedback_score: float | None
    flags: list[str]


class OnlineMonitor:
    """Simulates production online eval on sampled traces."""

    def __init__(self, sample_rate: float = 0.2) -> None:
        self.sample_rate = sample_rate
        self.traces: list[AgentTrace] = []
        self.feedback: dict[str, UserFeedback] = {}
        self.eval_results: list[OnlineEvalResult] = []

    def ingest_trace(self, trace: AgentTrace) -> None:
        self.traces.append(trace)
        if random.random() <= self.sample_rate:
            self._eval_trace(trace)

    def _eval_trace(self, trace: AgentTrace) -> None:
        flags: list[str] = []
        auto = 1.0
        if trace.error:
            auto, flags = 0.0, flags + ["error"]
        if trace.intent == "policy" and "pol-" not in trace.answer:
            auto, flags = min(auto, 0.3), flags + ["missing_citation"]
        if trace.latency_ms > 3000:
            flags.append("slow")
        fb = self.feedback.get(trace.trace_id)
        self.eval_results.append(OnlineEvalResult(
            trace_id=trace.trace_id, auto_score=auto,
            feedback_score=fb.score if fb else None, flags=flags,
        ))

    def record_feedback(self, fb: UserFeedback) -> None:
        self.feedback[fb.trace_id] = fb

    def satisfaction_rate(self) -> float:
        scores = [r.feedback_score for r in self.eval_results if r.feedback_score is not None]
        return sum(scores) / len(scores) if scores else 0.0

    def alert_rate(self) -> float:
        if not self.eval_results:
            return 0.0
        return sum(1 for r in self.eval_results if r.auto_score < 0.5 or r.flags) / len(self.eval_results)

---

## 7. Simulate Production Traffic

In [ ]:
PROD_QUERIES = [
    "What is the return policy?",
    "ORD-1002 status please",
    "How can I contact support?",
    "Return window for ORD-1001",
    "Do you sell cars?",
    "Refund timing?",
    "Where is my order ORD-1001",
    "Shipping cost?",
]

monitor = OnlineMonitor(sample_rate=0.5)
for i, q in enumerate(PROD_QUERIES):
    trace = PROD_AGENT.invoke(q, user_id=f"user-{i}", session_id=f"sess-{i}")
    monitor.ingest_trace(trace)
    if i == 1:
        monitor.record_feedback(UserFeedback(trace.trace_id, 1.0, "helpful"))
    if i == 4:
        monitor.record_feedback(UserFeedback(trace.trace_id, 0.0, "unhelpful"))

print(f"Traces: {len(monitor.traces)}, sampled evals: {len(monitor.eval_results)}")
print(f"Alert rate: {monitor.alert_rate():.0%}, satisfaction: {monitor.satisfaction_rate():.0%}")
for ev in monitor.eval_results:
    if ev.flags or (ev.feedback_score is not None and ev.feedback_score < 0.5):
        print(f"  ⚠ {ev.trace_id}: auto={ev.auto_score} flags={ev.flags} fb={ev.feedback_score}")

---

## 8. Drift Detection — Online Metrics Over Time

In [ ]:
@dataclass
class MetricWindow:
    period: str
    trace_count: int
    mean_auto_score: float
    error_rate: float
    citation_miss_rate: float


def compute_window(traces: list[AgentTrace], evals: list[OnlineEvalResult], label: str) -> MetricWindow:
    if not traces:
        return MetricWindow(label, 0, 0.0, 0.0, 0.0)
    eval_by_id = {e.trace_id: e for e in evals}
    scores = [eval_by_id[t.trace_id].auto_score for t in traces if t.trace_id in eval_by_id]
    errors = sum(1 for t in traces if t.error)
    cite_miss = sum(1 for e in evals if "missing_citation" in e.flags)
    return MetricWindow(
        label, len(traces),
        sum(scores) / len(scores) if scores else 0.0,
        errors / len(traces),
        cite_miss / len(evals) if evals else 0.0,
    )


def detect_drift(current: MetricWindow, baseline: MetricWindow, *, score_drop: float = 0.15) -> list[str]:
    alerts: list[str] = []
    if baseline.mean_auto_score - current.mean_auto_score > score_drop:
        alerts.append(f"auto_score dropped {baseline.mean_auto_score:.2f} → {current.mean_auto_score:.2f}")
    if current.error_rate > baseline.error_rate + 0.05:
        alerts.append(f"error_rate rose to {current.error_rate:.0%}")
    if current.citation_miss_rate > baseline.citation_miss_rate + 0.1:
        alerts.append(f"citation_miss_rate rose to {current.citation_miss_rate:.0%}")
    return alerts


baseline_window = MetricWindow("week-1", 100, 0.92, 0.01, 0.02)
current_window = compute_window(monitor.traces, monitor.eval_results, "today")
drift_alerts = detect_drift(current_window, baseline_window)
print("Current window:", current_window)
print("Drift alerts:", drift_alerts or "none")

---

## 9. Shadow Mode — Compare Candidate vs Production

In [ ]:
@dataclass
class ShadowComparison:
    question: str
    prod_answer: str
    candidate_answer: str
    answers_match: bool
    prod_score: float
    candidate_score: float


def shadow_eval(prod: ShopCoAgent, candidate: ShopCoAgent, queries: list[str]) -> list[ShadowComparison]:
    results: list[ShadowComparison] = []
    for q in queries:
        pt = prod.invoke(q)
        ct = candidate.invoke(q)
        prod_s = 1.0 if "pol-" in pt.answer or "status:" in pt.answer.lower() else 0.5
        cand_s = 1.0 if "pol-" in ct.answer or "status:" in ct.answer.lower() else 0.5
        results.append(ShadowComparison(
            q, pt.answer[:80], ct.answer[:80],
            pt.answer.strip() == ct.answer.strip(), prod_s, cand_s,
        ))
    return results


CANDIDATE = ShopCoAgent(version="v1.1.0", broken=False)
shadows = shadow_eval(PROD_AGENT, CANDIDATE, ["Return policy?", "ORD-1001 status"])
for s in shadows:
    print(f"Q: {s.question}")
    print(f"  match={s.answers_match} prod={s.prod_score} candidate={s.candidate_score}")

shadows_broken = shadow_eval(PROD_AGENT, BROKEN_AGENT, ["Return policy?"])
print("\nBroken candidate on return policy:")
print(f"  prod: {shadows_broken[0].prod_answer}")
print(f"  candidate: {shadows_broken[0].candidate_answer}")

---

## 10. Feedback Loop — Production → Golden Set

In [ ]:
class GoldenSetManager:
    def __init__(self, cases: list[GoldenCase]) -> None:
        self.cases = list(cases)

    def promote_from_production(self, trace: AgentTrace, feedback: UserFeedback, reason: str) -> GoldenCase | None:
        if feedback.score >= 0.5:
            return None
        case_id = f"prod-{trace.trace_id[:8]}"
        if any(c.id == case_id for c in self.cases):
            return None
        case = GoldenCase(
            id=case_id, input=trace.question,
            must_contain=[w for w in trace.question.split() if len(w) > 4][:2],
            expected_intent=trace.intent,
            tags=["from_production", reason],
        )
        self.cases.append(case)
        return case


gsm = GoldenSetManager(GOLDEN_SET.copy())
bad_trace = monitor.traces[4]
bad_fb = monitor.feedback[bad_trace.trace_id]
new_case = gsm.promote_from_production(bad_trace, bad_fb, "user_thumbs_down")
print(f"Golden set grew: {len(GOLDEN_SET)} → {len(gsm.cases)}")
if new_case:
    print(f"Added: {new_case.id} tags={new_case.tags}")

---

## 11. ReleaseFlow — Offline Gate + Canary Online Check

In [ ]:
@dataclass
class ReleaseDecision:
    version: str
    offline_passed: bool
    shadow_passed: bool
    canary_online_ok: bool
    approved: bool
    reasons: list[str]


def release_gate(candidate: ShopCoAgent, golden: list[GoldenCase], shadow_queries: list[str]) -> ReleaseDecision:
    reasons: list[str] = []
    offline = run_offline_eval(candidate, golden, OFFLINE_BASELINE)
    if not offline.gate_passed:
        reasons.append(f"offline pass_rate {offline.pass_rate:.0%}")

    shadows = shadow_eval(PROD_AGENT, candidate, shadow_queries)
    shadow_ok = all(s.candidate_score >= s.prod_score - 0.1 for s in shadows)
    if not shadow_ok:
        reasons.append("shadow regression detected")

    canary_monitor = OnlineMonitor(sample_rate=1.0)
    for q in shadow_queries[:2]:
        canary_monitor.ingest_trace(candidate.invoke(q))
    canary_ok = canary_monitor.alert_rate() < 0.5
    if not canary_ok:
        reasons.append(f"canary alert_rate {canary_monitor.alert_rate():.0%}")

    approved = offline.gate_passed and shadow_ok and canary_ok
    return ReleaseDecision(
        candidate.version, offline.gate_passed, shadow_ok, canary_ok, approved, reasons,
    )


for agent, label in [(CANDIDATE, "good v1.1.0"), (BROKEN_AGENT, "broken v1.1.0")]:
    d = release_gate(agent, gsm.cases, ["Return policy?", "ORD-1001 status"])
    print(f"{label}: approved={d.approved} reasons={d.reasons or 'none'}")

---

## 12. Sampling Strategies for Online Eval

In [ ]:
class SamplingStrategy(str, Enum):
    RANDOM = "random"
    ERRORS_ONLY = "errors_only"
    LOW_CONFIDENCE = "low_confidence"
    NEW_INTENT = "new_intent"


def should_sample(trace: AgentTrace, strategy: SamplingStrategy, rate: float) -> bool:
    if strategy == SamplingStrategy.ERRORS_ONLY:
        return bool(trace.error)
    if strategy == SamplingStrategy.LOW_CONFIDENCE:
        return trace.intent == "general" and "help" in trace.answer.lower()
    if strategy == SamplingStrategy.NEW_INTENT:
        return trace.intent == "general"
    return random.random() <= rate


for strat in SamplingStrategy:
    n = sum(1 for t in monitor.traces if should_sample(t, strat, 0.3))
    print(f"{strat.value:16} would sample {n}/{len(monitor.traces)} traces")

---

## 13. Unified Quality Dashboard (Offline + Online)

In [ ]:
def quality_dashboard(offline: OfflineReport, monitor: OnlineMonitor, drift: list[str]) -> dict[str, Any]:
    return {
        "offline": {
            "last_run": offline.run_id,
            "pass_rate": offline.pass_rate,
            "gate": offline.gate_passed,
            "agent_version": offline.agent_version,
        },
        "online": {
            "traces_24h": len(monitor.traces),
            "sampled_evals": len(monitor.eval_results),
            "alert_rate": round(monitor.alert_rate(), 3),
            "satisfaction": round(monitor.satisfaction_rate(), 3),
        },
        "drift_alerts": drift,
        "golden_set_size": len(gsm.cases),
    }


print(pretty(quality_dashboard(offline_ok, monitor, drift_alerts)))

---

## 14. Eval Harness

In [ ]:
EVAL = [
    ("offline gate passes prod", lambda: offline_ok.gate_passed),
    ("offline blocks broken", lambda: not offline_bad.gate_passed),
    ("online samples traces", lambda: len(monitor.eval_results) > 0),
    ("feedback recorded", lambda: len(monitor.feedback) >= 1),
    ("golden promotion", lambda: len(gsm.cases) > len(GOLDEN_SET)),
    ("release blocks broken", lambda: not release_gate(BROKEN_AGENT, gsm.cases, ["Return policy?"]).approved),
    ("release approves good", lambda: release_gate(CANDIDATE, gsm.cases, ["Return policy?", "ORD-1001 status"]).approved),
]

passed = sum(int(fn()) for _, fn in EVAL)
for label, fn in EVAL:
    print(f"{'✓' if fn() else '✗'} {label}")
print(f"\nEval: {passed}/{len(EVAL)}")

---

## 15. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Offline only | Miss production edge cases | Add online sampling |
| Online only | Ship regressions to prod | CI golden gate |
| 100% online LLM judge | Cost explosion | Sample 5–20% |
| No feedback loop | Same failures repeat | Promote bad traces to golden |
| Skip shadow mode | Bad canary hits users | Shadow compare before rollout |

---

## 16. Production Checklist

- [ ] Offline golden set in CI with pass-rate gate
- [ ] Online trace ingestion (Langfuse / LangSmith)
- [ ] Sampled auto-eval on production traces
- [ ] User feedback (thumbs) linked to trace_id
- [ ] Drift alerts on auto_score and error_rate
- [ ] Shadow mode before full rollout
- [ ] Process to promote failures → golden set
- [ ] Unified dashboard: offline + online metrics

---

## 17. Optional Live LLM Judge on Sampled Traces

In [ ]:
def online_llm_judge(trace: AgentTrace) -> float:
    if not USE_LIVE_LLM:
        return 1.0 if trace.citations or "status:" in trace.answer.lower() else 0.6

    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    prompt = f"Score 0-1 if this support answer is helpful and grounded.\nQ: {trace.question}\nA: {trace.answer}\nNumber only:"
    try:
        return float(llm.invoke(prompt).content.strip())
    except ValueError:
        return 0.5


sample_trace = PROD_AGENT.invoke("Return policy?")
print(f"Online judge score: {online_llm_judge(sample_trace):.2f}")

---

## 18. Quiz

1. What is the main purpose of offline vs online evaluation?
2. Why sample online eval instead of judging every trace?
3. What is shadow mode and when do you use it?
4. How does the feedback loop improve the golden set?
5. What metrics would trigger a drift alert?

---

## 19. Summary

**Offline evaluation** gates releases with deterministic golden tests in CI. **Online evaluation** monitors production traces, user feedback, and drift.

1. **Offline** — `run_offline_eval` + pass-rate baseline → deploy gate
2. **Online** — `OnlineMonitor` samples traces, auto-scores, collects feedback
3. **Shadow** — compare candidate vs prod without user impact
4. **Feedback loop** — thumbs-down traces become new golden cases
5. **ReleaseFlow** — offline + shadow + canary before approval

Use both modes together: offline prevents known regressions; online discovers unknown ones.